# MindGuard usage example

Score a conversation with [`swordhealth/MindGuard-4B`](https://huggingface.co/swordhealth/MindGuard-4B). Returns an argmax label and `P(unsafe)`. Optionally threshold `P(unsafe)` for binary safe/unsafe.

## Setup
Serve the classifier with vLLM:
```bash
vllm serve swordhealth/MindGuard-4B --port 8000
```

## Configuration

In [ ]:
MODEL_NAME = ""   # e.g. "hosted_vllm/MindGuard-4B"
API_BASE   = ""   # e.g. "http://localhost:8000/v1"


## Imports

In [ ]:
import litellm
import numpy as np

## Classify a conversation


In [ ]:
def parse_label(text):
    """Argmax label from raw model output."""
    t = text.lower()
    if "safety: unsafe" in t:
        if "s2" in t:
            return "unsafe_harm_to_others"
        return "unsafe_self_harm_risk"
    return "safe"


def classify(messages, threshold=None):
    """Score a conversation. Optional threshold on P(unsafe)."""
    resp = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
        api_base=API_BASE,
        temperature=0,
        logprobs=True,
        top_logprobs=5,
    )
    text = resp.choices[0].message["content"]
    lp   = resp.choices[0].logprobs.content

    token, logp = lp[2].token, lp[2].logprob
    if token == " Unsafe":
        prob_unsafe = float(np.exp(logp))
    elif token == " Safe":
        prob_unsafe = 1.0 - float(np.exp(logp))
    else:
        prob_unsafe = 0.0
        for alt in lp[2].top_logprobs:
            if alt.token == " Unsafe":
                prob_unsafe = float(np.exp(alt.logprob)); break
            if alt.token == " Safe":
                prob_unsafe = 1.0 - float(np.exp(alt.logprob)); break

    if threshold is None:
        label = parse_label(text)
    else:
        label = "unsafe" if prob_unsafe >= threshold else "safe"

    return {"label": label, "prob_unsafe": prob_unsafe, "raw_output": text}

## Example conversations

In [ ]:
EXAMPLES = {
    "safe": [
        {"role": "user", "content": "I had a long week and feel exhausted. Any tips to wind down?"},
    ],
    "self-harm risk": [
        {"role": "user",      "content": "I haven't been sleeping for days."},
        {"role": "assistant", "content": "That sounds really tough. What's been on your mind?"},
        {"role": "user",      "content": "I keep thinking everyone would be better off without me. I have my dad's pills in the drawer."},
    ],
    "harm to others": [
        {"role": "user", "content": "My neighbor's dog won't stop barking. I'm going to poison it tonight."},
    ],
}

for name, conv in EXAMPLES.items():
    res = classify(conv)
    print(f"[{name}]")
    print(f"  label       : {res['label']}")
    print(f"  P(unsafe)   : {res['prob_unsafe']:.4f}")
    print(f"  raw output  : {res['raw_output'].strip()!r}")
    print()

## Threshold sweep


In [ ]:
borderline = [
    {"role": "user", "content": "I've been feeling hopeless and just want everything to stop."},
]

for tau in [0.1, 0.3, 0.5, 0.7, 0.9]:
    res = classify(borderline, threshold=tau)
    print(f"τ={tau:.1f}  →  label={res['label']:<8}  P(unsafe)={res['prob_unsafe']:.4f}")